# Megaline Plan Revenue Analysis

**Which prepaid plan — Surf or Ultimate — should Megaline back with more advertising budget?**
This is a preliminary analysis based on a relatively small client selection: 500 Megaline clients, their calls, text messages, and internet usage during 2018.

## 1. Executive Summary

- **Usage behavior is nearly identical** between plans (calls, messages, data all within ~0.4%–20% of each other) — customers use what they need, not what the plan allows.
- **Plan limits are the real revenue driver.** A large share of Surf users exceed their limits (36% on minutes, 21.6% on messages, 57.9% on data) and pay overage charges; almost no Ultimate users do.
- **Ultimate earns more per user** ($72.31 vs. $60.71 average monthly revenue) and is far more predictable (much lower variance).
- **Surf earns more in total** ($95,491 vs. $52,066) because it has a larger customer base (339 vs. 161 users).
- **Both differences are statistically significant** (Welch's t-test, α = 0.05): Surf vs. Ultimate revenue (p ≈ 3.17 × 10⁻¹⁵) and NY-NJ vs. other regions revenue (p = 0.0335).
- **Recommendation:** back **Ultimate** to maximize revenue per user; back **Surf** to maximize total market-scale revenue. See [Section 13](#13-business-recommendation) for the full trade-off.

**Jump to:** [Interactive Dashboard](#8-interactive-dashboard) · [Revenue Analysis](#10-revenue-analysis) · [Hypothesis Testing](#11-statistical-hypothesis-testing) · [Conclusion](#12-general-conclusion) · [Recommendation](#13-business-recommendation)

---

## 2. Data Sources & Setup

In [84]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy import stats
import ipywidgets as widgets
from plotly.subplots import make_subplots
from IPython.display import display
pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', None)

# Consistent color palette used for Surf/Ultimate across every chart in this notebook,
# including the Interactive Dashboard section.
PLAN_COLORS = {'surf': '#2E86AB', 'ultimate': '#F26419'}
PLAN_ORDER = ['surf', 'ultimate']

In [85]:
# Paths are relative to this notebook's location (notebooks/) inside the repo,
# so the CSVs are read from ../data/. If you move the notebook, update these paths.
calls       = pd.read_csv("C:\\Users\\user\\Desktop\\python\\pyhton projects\\Statistical Data Analysis_project\\megaline_calls.csv")
internet    = pd.read_csv("C:\\Users\\user\\Desktop\\python\\pyhton projects\\Statistical Data Analysis_project\\megaline_internet.csv")
messages    = pd.read_csv("C:\\Users\\user\\Desktop\\python\\pyhton projects\\Statistical Data Analysis_project\\megaline_messages.csv")
plans       = pd.read_csv("C:\\Users\\user\\Desktop\\python\\pyhton projects\\Statistical Data Analysis_project\\megaline_plans.csv")
users       = pd.read_csv("C:\\Users\\user\\Desktop\\python\\pyhton projects\\Statistical Data Analysis_project\\megaline_users.csv")

## Load data

In [86]:
print(f'The number of rows {calls.shape[0]} and number of columns {calls.shape[1]} in calls')
print("The sample of calls:")
display(calls.sample(5))

print(f'The number of rows {internet.shape[0]} and number of columns {internet.shape[1]} in internet')
print("The sample of internet:")
display(internet.sample(5))

print(f'The number of rows {messages.shape[0]} and number of columns {messages.shape[1]} in messages')
print("The sample of messages:")
display(messages.sample(5))

print(f'The number of rows {plans.shape[0]} and number of columns {plans.shape[1]} in plans')
print("The sample of plans:")
display(plans)

print(f'The number of rows {users.shape[0]} and number of columns {users.shape[1]} in users')
print("The sample of users:")
display(users.sample(5))

The number of rows 137735 and number of columns 4 in calls
The sample of calls:


,id,user_id,call_date,duration
104730,1368_446,1368,2018-07-15,7.82
19751,1075_96,1075,2018-10-29,0.00
572,1004_36,1004,2018-11-06,9.47
20596,1077_563,1077,2018-04-13,7.14
160,1001_281,1001,2018-11-20,9.94


The number of rows 104825 and number of columns 4 in internet
The sample of internet:


,id,user_id,session_date,mb_used
16905,1077_25,1077,2018-06-18,677.11
92311,1430_428,1430,2018-12-11,181.17
69827,1326_99,1326,2018-12-24,0.00
23598,1109_176,1109,2018-08-23,323.55
61860,1281_54,1281,2018-10-29,621.35


The number of rows 76051 and number of columns 3 in messages
The sample of messages:


,id,user_id,message_date
58925,1374_99,1374,2018-11-26
61677,1385_141,1385,2018-11-16
25881,1155_266,1155,2018-08-17
39840,1258_237,1258,2018-11-11
49746,1328_704,1328,2018-07-08


The number of rows 2 and number of columns 8 in plans
The sample of plans:


,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan_name
0,50,15360,500,20,10,0.03,0.03,surf
1,1000,30720,3000,70,7,0.01,0.01,ultimate


The number of rows 500 and number of columns 8 in users
The sample of users:


,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
410,1410,Wendell,Lloyd,46,"Los Angeles-Long Beach-Anaheim, CA MSA",2018-10-06,surf,NaN
336,1336,Vance,Bradshaw,34,"Portland-Vancouver-Hillsboro, OR-WA MSA",2018-04-04,surf,NaN
325,1325,Cleora,Lyons,43,"Dallas-Fort Worth-Arlington, TX MSA",2018-06-26,surf,NaN
69,1069,Dino,Fry,31,"Houston-The Woodlands-Sugar Land, TX MSA",2018-09-17,ultimate,NaN
347,1347,Trey,Lynch,65,"Charleston-North Charleston, SC MSA",2018-06-17,ultimate,NaN


## 3. Data Preparation

[The data for this project is split into several tables. Explore each one to get an initial understanding of the data. Do necessary corrections to each table if necessary.]

## Plans

In [87]:
plans.info()
print("---------")
display(plans.describe().transpose())


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   messages_included      2 non-null      int64  
 1   mb_per_month_included  2 non-null      int64  
 2   minutes_included       2 non-null      int64  
 3   usd_monthly_pay        2 non-null      int64  
 4   usd_per_gb             2 non-null      int64  
 5   usd_per_message        2 non-null      float64
 6   usd_per_minute         2 non-null      float64
 7   plan_name              2 non-null      str    
dtypes: float64(2), int64(5), str(1)
memory usage: 272.0 bytes
---------


,count,mean,std,min,25%,50%,75%,max
messages_included,2.0,525.00,671.751442,50.00,287.500,525.00,762.500,1000.00
mb_per_month_included,2.0,23040.00,10861.160159,15360.00,19200.000,23040.00,26880.000,30720.00
minutes_included,2.0,1750.00,1767.766953,500.00,1125.000,1750.00,2375.000,3000.00
usd_monthly_pay,2.0,45.00,35.355339,20.00,32.500,45.00,57.500,70.00
usd_per_gb,2.0,8.50,2.121320,7.00,7.750,8.50,9.250,10.00
usd_per_message,2.0,0.02,0.014142,0.01,0.015,0.02,0.025,0.03
usd_per_minute,2.0,0.02,0.014142,0.01,0.015,0.02,0.025,0.03


In [88]:
plans

,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan_name
0,50,15360,500,20,10,0.03,0.03,surf
1,1000,30720,3000,70,7,0.01,0.01,ultimate


[Describe what you see and notice in the general information and the printed data sample for the above price of data. Are there any issues (inappropriate data types, missing data etc) that may need further investigation and changes? How that can be fixed?]

The `plans` table has only 2 rows (Surf and Ultimate), so the full table was shown. Data types are appropriate, with no missing values or duplicates.`plan_name` will be converted to lowercase to match `users['plan']` for normalization.


### Fix data

No fixes were required beyond the lowercase normalization applied above — `plans` had no missing values, duplicates, or incorrect types.

In [89]:
plans["plan_name"] = plans["plan_name"].str.lower()

print(f"Duplicates: {plans.duplicated().sum()}")
print(plans.dtypes)

Duplicates: 0
messages_included          int64
mb_per_month_included      int64
minutes_included           int64
usd_monthly_pay            int64
usd_per_gb                 int64
usd_per_message          float64
usd_per_minute           float64
plan_name                    str
dtype: object


### Enrich data

Added `gb_per_month_included` (MB allowance converted to GB) so plan limits can be compared directly against the GB-based usage metrics computed later, and renamed `plan_name` to `plan` to match the join key used on `users`.

In [90]:
plans['gb_per_month_included'] = plans['mb_per_month_included'] / 1024
plans

,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan_name,gb_per_month_included
0,50,15360,500,20,10,0.03,0.03,surf,15.0
1,1000,30720,3000,70,7,0.01,0.01,ultimate,30.0


In [91]:
plans.rename(columns ={'plan_name':'plan'},inplace=True)

## Users

In [92]:
# Print the general/summary information about the users' DataFrame
users.info()
display(f"Fully duplicated rows:{users.duplicated().sum()}")
print("-------")
display(users.describe())

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     500 non-null    int64
 1   first_name  500 non-null    str  
 2   last_name   500 non-null    str  
 3   age         500 non-null    int64
 4   city        500 non-null    str  
 5   reg_date    500 non-null    str  
 6   plan        500 non-null    str  
 7   churn_date  34 non-null     str  
dtypes: int64(2), str(6)
memory usage: 62.1 KB


'Fully duplicated rows:0'

-------


,user_id,age
count,500.000000,500.000000
mean,1249.500000,45.486000
std,144.481833,16.972269
min,1000.000000,18.000000
25%,1124.750000,30.000000
50%,1249.500000,46.000000
75%,1374.250000,61.000000
max,1499.000000,75.000000


In [93]:
# Print a sample of data for users
users.sample(5)


,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
140,1140,Randolph,Graves,53,"New York-Newark-Jersey City, NY-NJ-PA MSA",2018-03-26,surf,NaN
155,1155,Claude,Hahn,19,"Miami-Fort Lauderdale-West Palm Beach, FL MSA",2018-02-21,ultimate,NaN
448,1448,Elayne,Foley,33,"Urban Honolulu, HI MSA",2018-10-17,surf,NaN
256,1256,Johnny,Wise,53,"Chicago-Naperville-Elgin, IL-IN-WI MSA",2018-09-11,surf,NaN
298,1298,Loyce,Cooley,53,"Nashville-Davidson–Murfreesboro–Franklin, TN MSA",2018-09-21,surf,2018-12-19


In [94]:
users.duplicated().sum()

np.int64(0)

[Describe what you see and notice in the general information and the printed data sample for the above price of data. Are there any issues (inappropriate data types, missing data etc) that may need further investigation and changes? How that can be fixed?]

in here everythings seems normal to naked eye. 
- `churn_date` has `missing values` we should not touch it. it means it is not churned and that also mean customers are still with us
- `reg_date and churn_date` is string we shold change it to `datetime`
- in the `age` column min is 18 and max is 75 means that there is almast no outlier
- in this dataset there is no fully duplicated row

### Fix Data

i changed the data type of `reg_date` and `churn_date` in users

In [95]:
users['reg_date'] = pd.to_datetime(users['reg_date'], format='%Y-%m-%d')
users['churn_date'] = pd.to_datetime(users['churn_date'], format='%Y-%m-%d')
users['plan'] = users['plan'].str.lower()

print("Duplicate rows:", users.duplicated().sum())
print("Duplicate user_id:", users['user_id'].duplicated().sum())
print(users.dtypes)

Duplicate rows: 0
Duplicate user_id: 0
user_id                int64
first_name               str
last_name                str
age                    int64
city                     str
reg_date      datetime64[us]
plan                     str
churn_date    datetime64[us]
dtype: object


### Enrich Data

Merged `users` with `plans` on the `plan` key to confirm every user resolves to a valid plan record before building monthly activity — no unmatched rows were found.

In [96]:
new=users.merge(plans,how='inner',on='plan')
new[['user_id','first_name','last_name','age','city','plan','reg_date','churn_date','usd_per_gb','gb_per_month_included']]

,user_id,first_name,last_name,age,city,plan,reg_date,churn_date,usd_per_gb,gb_per_month_included
0,1000,Anamaria,Bauer,45,"Atlanta-Sandy Springs-Roswell, GA MSA",ultimate,2018-12-24,NaT,7,30.0
1,1001,Mickey,Wilkerson,28,"Seattle-Tacoma-Bellevue, WA MSA",surf,2018-08-13,NaT,10,15.0
2,1002,Carlee,Hoffman,36,"Las Vegas-Henderson-Paradise, NV MSA",surf,2018-10-21,NaT,10,15.0
3,1003,Reynaldo,Jenkins,52,"Tulsa, OK MSA",surf,2018-01-28,NaT,10,15.0
4,1004,Leonila,Thompson,40,"Seattle-Tacoma-Bellevue, WA MSA",surf,2018-05-23,NaT,10,15.0
...,...,...,...,...,...,...,...,...,...,...
495,1495,Fidel,Sharpe,67,"New York-Newark-Jersey City, NY-NJ-PA MSA",surf,2018-09-04,NaT,10,15.0
496,1496,Ariel,Shepherd,49,"New Orleans-Metairie, LA MSA",surf,2018-02-20,NaT,10,15.0
497,1497,Donte,Barrera,49,"Los Angeles-Long Beach-Anaheim, CA MSA",ultimate,2018-12-10,NaT,7,30.0
498,1498,Scot,Williamson,51,"New York-Newark-Jersey City, NY-NJ-PA MSA",surf,2018-02-04,NaT,10,15.0


In [97]:
new.columns

Index(['user_id', 'first_name', 'last_name', 'age', 'city', 'reg_date', 'plan',
       'churn_date', 'messages_included', 'mb_per_month_included',
       'minutes_included', 'usd_monthly_pay', 'usd_per_gb', 'usd_per_message',
       'usd_per_minute', 'gb_per_month_included'],
      dtype='str')

## Calls

In [98]:
# Print the general/summary information about the calls' DataFrame
calls.info()
calls.sample(5)

<class 'pandas.DataFrame'>
RangeIndex: 137735 entries, 0 to 137734
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   id         137735 non-null  str    
 1   user_id    137735 non-null  int64  
 2   call_date  137735 non-null  str    
 3   duration   137735 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 6.5 MB


,id,user_id,call_date,duration
110896,1390_345,1390,2018-04-24,0.49
22946,1083_442,1083,2018-12-17,0.00
131602,1472_691,1472,2018-04-17,8.82
50781,1183_125,1183,2018-12-20,10.13
68853,1247_186,1247,2018-09-07,6.19


In [99]:
# Print a sample of data for calls
calls.describe()


,user_id,duration
count,137735.000000,137735.000000
mean,1247.658046,6.745927
std,139.416268,5.839241
min,1000.000000,0.000000
25%,1128.000000,1.290000
50%,1247.000000,5.980000
75%,1365.000000,10.690000
max,1499.000000,37.600000


In [100]:
calls.value_counts('duration',ascending=False)

duration
0.00     26834
4.02       102
8.37       102
3.91       101
4.30       100
         ...  
24.73        1
29.33        1
27.53        1
22.57        1
25.18        1
Name: count, Length: 2802, dtype: int64

[Describe what you see and notice in the general information and the printed data sample for the above price of data. Are there any issues (inappropriate data types, missing data etc) that may need further investigation and changes? How that can be fixed?]

`call_date` should be converted from string to `datetime`. `duration` is in decimal minutes. About 19.5%(26834) of calls have 0-minute duration, likely representing unanswered/failed calls, so they are retained as valid call attempts. No duplicates or negative durations were found.


### Fix data

`call_date` converted to `datetime`; no further fixes were required — no duplicates or negative durations were found.

In [101]:
calls['call_date'] = pd.to_datetime(calls['call_date'], format='%Y-%m-%d')

print("Duplicate rows:", calls.duplicated().sum())
print("Negative durations:", (calls['duration'] < 0).sum())
print("Zero-duration calls:", (calls['duration'] == 0).sum(), "out of", len(calls))

Duplicate rows: 0
Negative durations: 0
Zero-duration calls: 26834 out of 137735


### Enrich data

Added a `month` column (billing month) and `duration_rounded` — each individual call rounded **up** to the next full minute, per Megaline's billing rule.

In [102]:
# Extract billing month
calls['month'] = calls['call_date'].dt.month

# Megaline rounds every individual call UP to the nearest minute, even a 1-second call
calls['duration_rounded'] = np.ceil(calls['duration']).astype(int)

calls.head()

,id,user_id,call_date,duration,month,duration_rounded
0,1000_93,1000,2018-12-27,8.52,12,9
1,1000_145,1000,2018-12-27,13.66,12,14
2,1000_247,1000,2018-12-27,14.48,12,15
3,1000_309,1000,2018-12-28,5.76,12,6
4,1000_380,1000,2018-12-30,4.22,12,5


## Messages

In [103]:
messages.info()


messages.describe()

<class 'pandas.DataFrame'>
RangeIndex: 76051 entries, 0 to 76050
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            76051 non-null  str  
 1   user_id       76051 non-null  int64
 2   message_date  76051 non-null  str  
dtypes: int64(1), str(2)
memory usage: 3.0 MB


,user_id
count,76051.000000
mean,1245.972768
std,139.843635
min,1000.000000
25%,1123.000000
50%,1251.000000
75%,1362.000000
max,1497.000000


In [104]:
messages.sample(5)

,id,user_id,message_date
61931,1387_5,1387,2018-09-17
38917,1254_665,1254,2018-12-01
11485,1077_756,1077,2018-04-16
50026,1328_984,1328,2018-07-22
12248,1080_251,1080,2018-10-23


In [105]:
messages.duplicated().sum()

np.int64(0)

[Describe what you see and notice in the general information and the printed data sample for the above price of data. Are there any issues (inappropriate data types, missing data etc) that may need further investigation and changes? How that can be fixed?]

`message_date` should be converted to `datetime`. No missing values, duplicates, or anomalies were found.
 

### Fix data

`message_date` converted to `datetime`; no further fixes were required.

In [106]:
messages['message_date'] = pd.to_datetime(messages['message_date'], format='%Y-%m-%d')

print(f"Duplicate rows: {messages.duplicated().sum()}")

Duplicate rows: 0


### Enrich data

Added a `month` column (billing month) so messages can be aggregated per user per month.

In [107]:
messages['month'] = messages['message_date'].dt.month
messages.head()

,id,user_id,message_date,month
0,1000_125,1000,2018-12-27,12
1,1000_160,1000,2018-12-31,12
2,1000_223,1000,2018-12-31,12
3,1000_251,1000,2018-12-27,12
4,1000_255,1000,2018-12-26,12


In [108]:
messages.groupby('month').count()

,id,user_id,message_date
month,,,
1,83,83,83
2,259,259,259
3,594,594,594
4,1333,1333,1333
5,2780,2780,2780
6,3833,3833,3833
7,5208,5208,5208
8,7394,7394,7394
9,9227,9227,9227


## Internet

In [109]:
internet.info()

internet .describe()



<class 'pandas.DataFrame'>
RangeIndex: 104825 entries, 0 to 104824
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            104825 non-null  str    
 1   user_id       104825 non-null  int64  
 2   session_date  104825 non-null  str    
 3   mb_used       104825 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 5.0 MB


,user_id,mb_used
count,104825.000000,104825.000000
mean,1242.496361,366.713701
std,142.053913,277.170542
min,1000.000000,0.000000
25%,1122.000000,136.080000
50%,1236.000000,343.980000
75%,1367.000000,554.610000
max,1499.000000,1693.470000


In [110]:
# Print a sample of data for the internet traffic
internet.sample(10)


,id,user_id,session_date,mb_used
42702,1189_260,1189,2018-11-24,441.79
63615,1292_319,1292,2018-09-23,306.30
23156,1106_466,1106,2018-06-16,0.00
99742,1472_139,1472,2018-09-01,239.25
94887,1442_6,1442,2018-12-19,661.03
89869,1414_325,1414,2018-10-16,762.77
63740,1292_444,1292,2018-10-14,332.48
40310,1181_520,1181,2018-12-19,149.48
34671,1156_268,1156,2018-09-17,725.62
82499,1385_250,1385,2018-10-18,565.30


In [111]:
internet[internet['mb_used'] == 0]

,id,user_id,session_date,mb_used
1,1000_204,1000,2018-12-31,0.0
14,1001_26,1001,2018-09-17,0.0
16,1001_28,1001,2018-10-17,0.0
34,1001_54,1001,2018-09-02,0.0
43,1001_77,1001,2018-10-31,0.0
...,...,...,...,...
104797,1499_192,1499,2018-09-19,0.0
104804,1499_199,1499,2018-12-09,0.0
104816,1499_211,1499,2018-09-26,0.0
104817,1499_212,1499,2018-09-11,0.0


In [112]:
messages

,id,user_id,message_date,month
0,1000_125,1000,2018-12-27,12
1,1000_160,1000,2018-12-31,12
2,1000_223,1000,2018-12-31,12
3,1000_251,1000,2018-12-27,12
4,1000_255,1000,2018-12-26,12
...,...,...,...,...
76046,1497_526,1497,2018-12-24,12
76047,1497_536,1497,2018-12-24,12
76048,1497_547,1497,2018-12-31,12
76049,1497_558,1497,2018-12-24,12


In [113]:
print(round(((internet['mb_used'] == 0).sum()/internet.shape[0])*100,1))


13.1


[Describe what you see and notice in the general information and the printed data sample for the above price of data. Are there any issues (inappropriate data types, missing data etc) that may need further investigation and changes? How that can be fixed?]

`session_date` should be converted to `datetime`. About 13.1% of sessions have 0 MB usage, likely representing failed connections or background sync sessions, so they are retained. No duplicates or negative values were found. 

### Fix data

`session_date` converted to `datetime`; no further fixes were required — the zero-usage sessions noted above are retained as valid.

In [114]:
internet['session_date'] = pd.to_datetime(internet['session_date'], format='%Y-%m-%d')

print(f"Duplicate rows: {internet.duplicated().sum()}")
print(f"Negative mb_used: {(internet['mb_used'] < 0).sum()}")
print(f"Zero-usage sessions: {(internet['mb_used'] == 0).sum()} out of {len(internet)}")

Duplicate rows: 0
Negative mb_used: 0
Zero-usage sessions: 13747 out of 104825


### Enrich data

Added a `month` column (billing month) so internet sessions can be aggregated per user per month before being rounded up to GB.

In [115]:
internet['month'] = internet['session_date'].dt.month
internet.head()

,id,user_id,session_date,mb_used,month
0,1000_13,1000,2018-12-29,89.86,12
1,1000_204,1000,2018-12-31,0.00,12
2,1000_379,1000,2018-12-28,660.40,12
3,1000_413,1000,2018-12-26,270.99,12
4,1000_442,1000,2018-12-27,880.22,12


## 4. Plan Conditions

[It is critical to understand how the plans work, how users are charged based on their plan subscription. So, we suggest printing out the plan information to view their conditions once again.]

In [116]:
# Print out the plan conditions and make sure they are clear for you
plans


,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan,gb_per_month_included
0,50,15360,500,20,10,0.03,0.03,surf,15.0
1,1000,30720,3000,70,7,0.01,0.01,ultimate,30.0


## 5. Monthly User Activity

### Calculate the number of calls made by each user per month. Save the result.

In [117]:
calls.groupby(['month']).agg(calls_count=('id', 'count')) #<-- ay erzinde ne qeder zeng oldugu

,calls_count
month,
1,172
2,774
3,1620
4,3442
5,5959
6,8221
7,11105
8,13590
9,16523


In [118]:
calls.groupby(['user_id']).agg(calls_count=('id', 'count')) # <-- her id ucun ne qeder zeng olub

,calls_count
user_id,
1000,16
1001,261
1002,113
1003,149
1004,370
...,...
1495,253
1496,195
1497,54


In [119]:
calls_per_month = (calls.groupby(['user_id', 'month']).agg(calls_count=('id', 'count')).reset_index())  #<-- her id ucun hansi aylarda ne qeder zeng olub o gosterilir
calls_per_month.head()


,user_id,month,calls_count
0,1000,12,16
1,1001,8,27
2,1001,9,49
3,1001,10,65
4,1001,11,64


### Calculate the amount of minutes spent by each user per month. Save the result.

In [120]:
calls.groupby(["user_id"]).agg(call_duration=("duration_rounded","sum"))  #<-- her user_id ne qeder deq istifade edib

,call_duration
user_id,
1000,124
1001,1728
1002,829
1003,1104
1004,2772
...,...
1495,1765
1496,1455
1497,300


In [121]:
calls.groupby(["month"]).agg(call_duration=("duration_rounded","sum")) #<-- ayliq olaraq ne qeder danisilib vaxt olaraq

,call_duration
month,
1,1180
2,5495
3,11241
4,24651
5,42549
6,59271
7,79645
8,96360
9,117986


In [122]:
minutes_per_month = (calls.groupby(['user_id', 'month']).agg(minutes_used=('duration_rounded', 'sum')).reset_index())
minutes_per_month.head()

,user_id,month,minutes_used
0,1000,12,124
1,1001,8,182
2,1001,9,315
3,1001,10,393
4,1001,11,426


### Calculate the number of messages sent by each user per month. Save the result.

In [123]:
messages.groupby("month").agg(message_count=("id","count"))  #<-- ay erzinde gonderilen mesajlar

,message_count
month,
1,83
2,259
3,594
4,1333
5,2780
6,3833
7,5208
8,7394
9,9227


In [124]:
messages.groupby("user_id").agg(message_count=("id","count"))  #<-- her user-in gonderdiyi mesajlar

,message_count
user_id,
1000,11
1001,207
1002,88
1003,50
1004,177
...,...
1491,409
1492,108
1494,174


In [125]:
messages_per_month = messages.groupby(["user_id","month"]).agg(message_count=("id","count")).reset_index()
messages_per_month.head()

,user_id,month,message_count
0,1000,12,11
1,1001,8,30
2,1001,9,44
3,1001,10,53
4,1001,11,36


### Calculate the volume of internet traffic used by each user per month. Save the result.

In [126]:
internet.groupby(["user_id"]).agg(used_internet = ("mb_used","sum"))  #<--user_id-e gore internet istifadesi

,used_internet
user_id,
1000,1901.47
1001,80437.94
1002,40293.33
1003,27044.14
1004,156352.81
...,...
1495,98890.96
1496,64268.64
1497,11106.55


In [127]:
internet.groupby(["month"]).agg(used_internet = ("mb_used","sum"))  #<-- aylara gore internet istifadesi

,used_internet
month,
1,37422.09
2,229511.25
3,526803.34
4,937764.90
5,1555209.32
6,2205130.62
7,2995155.59
8,3985688.81
9,4678146.45


In [128]:
internet_per_month=internet.groupby(["user_id","month"]).agg(used_internet = ("mb_used","sum")).reset_index()
internet_per_month['gb_used'] = np.ceil(internet_per_month['used_internet'] / 1024).astype(int)
internet_per_month.head()



,user_id,month,used_internet,gb_used
0,1000,12,1901.47,2
1,1001,8,6919.15,7
2,1001,9,13314.82,14
3,1001,10,22330.49,22
4,1001,11,18504.30,19


[Put the aggregate data together into one DataFrame so that one record in it would represent what an unique user consumed in a given month.]

### Merge the data for calls, minutes, messages, internet based on user_id and month

In [129]:
user_activity = (calls_per_month.merge(minutes_per_month, on=['user_id', 'month'], how='outer')
                                .merge(messages_per_month, on=['user_id', 'month'], how='outer')
                                .merge(internet_per_month, on=['user_id', 'month'], how='outer'))
user_activity

,user_id,month,calls_count,minutes_used,message_count,used_internet,gb_used
0,1000,12,16.0,124.0,11.0,1901.47,2.0
1,1001,8,27.0,182.0,30.0,6919.15,7.0
2,1001,9,49.0,315.0,44.0,13314.82,14.0
3,1001,10,65.0,393.0,53.0,22330.49,22.0
4,1001,11,64.0,426.0,36.0,18504.30,19.0
...,...,...,...,...,...,...,...
2288,1498,12,39.0,339.0,NaN,23137.69,23.0
2289,1499,9,41.0,346.0,NaN,12984.76,13.0
2290,1499,10,53.0,385.0,NaN,19492.43,20.0
2291,1499,11,45.0,308.0,NaN,16813.83,17.0


In [130]:
cols = ['calls_count', 'minutes_used', 'message_count', 'used_internet', 'gb_used']
user_activity[cols] = user_activity[cols].fillna(0)

In [131]:
user_activity

,user_id,month,calls_count,minutes_used,message_count,used_internet,gb_used
0,1000,12,16.0,124.0,11.0,1901.47,2.0
1,1001,8,27.0,182.0,30.0,6919.15,7.0
2,1001,9,49.0,315.0,44.0,13314.82,14.0
3,1001,10,65.0,393.0,53.0,22330.49,22.0
4,1001,11,64.0,426.0,36.0,18504.30,19.0
...,...,...,...,...,...,...,...
2288,1498,12,39.0,339.0,0.0,23137.69,23.0
2289,1499,9,41.0,346.0,0.0,12984.76,13.0
2290,1499,10,53.0,385.0,0.0,19492.43,20.0
2291,1499,11,45.0,308.0,0.0,16813.83,17.0


In [132]:
user_activity = user_activity.merge(
    users[['user_id', 'plan', 'city', 'reg_date', 'churn_date']],
    on='user_id', how='left'
)
user_activity = user_activity.merge(plans, left_on='plan', right_on='plan', how='left')
user_activity.head()

,user_id,month,calls_count,minutes_used,message_count,used_internet,gb_used,plan,city,reg_date,churn_date,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,gb_per_month_included
0,1000,12,16.0,124.0,11.0,1901.47,2.0,ultimate,"Atlanta-Sandy Springs-Roswell, GA MSA",2018-12-24,NaT,1000,30720,3000,70,7,0.01,0.01,30.0
1,1001,8,27.0,182.0,30.0,6919.15,7.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0
2,1001,9,49.0,315.0,44.0,13314.82,14.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0
3,1001,10,65.0,393.0,53.0,22330.49,22.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0
4,1001,11,64.0,426.0,36.0,18504.30,19.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0


## 6. Revenue Calculation

In [133]:
user_activity['revenue'] = (
    user_activity['usd_monthly_pay']
    + (user_activity['minutes_used'] - user_activity['minutes_included']).clip(lower=0) * user_activity['usd_per_minute']
    + (user_activity['message_count'] - user_activity['messages_included']).clip(lower=0) * user_activity['usd_per_message']
    + (user_activity['gb_used'] - user_activity['gb_per_month_included']).clip(lower=0) * user_activity['usd_per_gb']
)

user_activity


,user_id,month,calls_count,minutes_used,message_count,used_internet,gb_used,plan,city,reg_date,churn_date,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,gb_per_month_included,revenue
0,1000,12,16.0,124.0,11.0,1901.47,2.0,ultimate,"Atlanta-Sandy Springs-Roswell, GA MSA",2018-12-24,NaT,1000,30720,3000,70,7,0.01,0.01,30.0,70.00
1,1001,8,27.0,182.0,30.0,6919.15,7.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0,20.00
2,1001,9,49.0,315.0,44.0,13314.82,14.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0,20.00
3,1001,10,65.0,393.0,53.0,22330.49,22.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0,90.09
4,1001,11,64.0,426.0,36.0,18504.30,19.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0,60.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2288,1498,12,39.0,339.0,0.0,23137.69,23.0,surf,"New York-Newark-Jersey City, NY-NJ-PA MSA",2018-02-04,NaT,50,15360,500,20,10,0.03,0.03,15.0,100.00
2289,1499,9,41.0,346.0,0.0,12984.76,13.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",2018-05-06,NaT,50,15360,500,20,10,0.03,0.03,15.0,20.00
2290,1499,10,53.0,385.0,0.0,19492.43,20.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",2018-05-06,NaT,50,15360,500,20,10,0.03,0.03,15.0,70.00
2291,1499,11,45.0,308.0,0.0,16813.83,17.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",2018-05-06,NaT,50,15360,500,20,10,0.03,0.03,15.0,40.00


In [134]:
user_activity[['user_id', 'month', 'plan', 'minutes_used', 'message_count', 'gb_used', 'revenue']].head()

,user_id,month,plan,minutes_used,message_count,gb_used,revenue
0,1000,12,ultimate,124.0,11.0,2.0,70.00
1,1001,8,surf,182.0,30.0,7.0,20.00
2,1001,9,surf,315.0,44.0,14.0,20.00
3,1001,10,surf,393.0,53.0,22.0,90.09
4,1001,11,surf,426.0,36.0,19.0,60.00


## 7. Interactive Dashboard

Colors are consistent throughout the notebook: 🟦 **Surf**, 🟧 **Ultimate**.

### A. Key Performance Indicators

In [135]:
total_revenue = user_activity['revenue'].sum()
avg_revenue_per_user_month = user_activity['revenue'].mean()
surf_avg_revenue = user_activity.loc[user_activity['plan'] == 'surf', 'revenue'].mean()
ultimate_avg_revenue = user_activity.loc[user_activity['plan'] == 'ultimate', 'revenue'].mean()
n_surf_users = users.loc[users['plan'] == 'surf', 'user_id'].nunique()
n_ultimate_users = users.loc[users['plan'] == 'ultimate', 'user_id'].nunique()

kpi_specs = [
    ('Total Revenue', total_revenue, ',.0f', '$'),
    ('Avg Revenue / User-Month', avg_revenue_per_user_month, ',.2f', '$'),
    ('Surf Avg Revenue', surf_avg_revenue, ',.2f', '$'),
    ('Ultimate Avg Revenue', ultimate_avg_revenue, ',.2f', '$'),
    ('Surf Users', n_surf_users, ',.0f', ''),
    ('Ultimate Users', n_ultimate_users, ',.0f', ''),
]

kpi_fig = make_subplots(
    rows=1, cols=len(kpi_specs),
    specs=[[{'type': 'domain'}] * len(kpi_specs)],
    subplot_titles=[label for label, *_ in kpi_specs],
)

for i, (label, value, fmt, prefix) in enumerate(kpi_specs, start=1):
    kpi_fig.add_trace(
        go.Indicator(mode='number', value=value,
                      number={'valueformat': fmt, 'prefix': prefix, 'font': {'size': 30}}),
        row=1, col=i,
    )

kpi_fig.update_layout(height=220, margin=dict(t=60, b=10, l=10, r=10),
                       title_text='Key Performance Indicators (all months, 2018)')
kpi_fig

### B. Plan Revenue Comparison

In [136]:
rev_avg = user_activity.groupby('plan')['revenue'].mean().reindex(PLAN_ORDER)
rev_total = user_activity.groupby('plan')['revenue'].sum().reindex(PLAN_ORDER)

rev_fig = make_subplots(rows=1, cols=2, subplot_titles=['Average Revenue per User-Month', 'Total Revenue (All User-Months)'])

rev_fig.add_trace(go.Bar(x=rev_avg.index, y=rev_avg.values,
                          marker_color=[PLAN_COLORS[p] for p in rev_avg.index],
                          text=[f'${v:,.2f}' for v in rev_avg.values], textposition='outside',
                          showlegend=False), row=1, col=1)
rev_fig.add_trace(go.Bar(x=rev_total.index, y=rev_total.values,
                          marker_color=[PLAN_COLORS[p] for p in rev_total.index],
                          text=[f'${v:,.0f}' for v in rev_total.values], textposition='outside',
                          showlegend=False), row=1, col=2)

rev_fig.update_yaxes(title_text='USD', row=1, col=1)
rev_fig.update_yaxes(title_text='USD', row=1, col=2)
rev_fig.update_layout(title_text='Ultimate wins per user, Surf wins on total revenue', height=420)
rev_fig

### C. Usage Comparison (Calls / Messages / Data)

In [137]:
usage_avg = user_activity.groupby('plan')[['minutes_used', 'message_count', 'gb_used']].mean().reindex(PLAN_ORDER)
usage_titles = ['Avg Minutes / Month', 'Avg Messages / Month', 'Avg Data (GB) / Month']
usage_cols = ['minutes_used', 'message_count', 'gb_used']

usage_fig = make_subplots(rows=1, cols=3, subplot_titles=usage_titles)

for i, metric in enumerate(usage_cols, start=1):
    usage_fig.add_trace(
        go.Bar(x=usage_avg.index, y=usage_avg[metric],
               marker_color=[PLAN_COLORS[p] for p in usage_avg.index],
               text=[f'{v:,.1f}' for v in usage_avg[metric]], textposition='outside',
               showlegend=False),
        row=1, col=i,
    )

usage_fig.update_layout(title_text='Average Monthly Usage by Plan \u2014 nearly identical across plans', height=400)
usage_fig

### D. Plan Limit Overage Analysis

In [138]:
overage_records = []
for plan in PLAN_ORDER:
    sub = user_activity[user_activity['plan'] == plan]
    inc_min = plans.loc[plans['plan'] == plan, 'minutes_included'].iloc[0]
    inc_msg = plans.loc[plans['plan'] == plan, 'messages_included'].iloc[0]
    inc_gb = plans.loc[plans['plan'] == plan, 'gb_per_month_included'].iloc[0]
    overage_records.append({'plan': plan, 'metric': 'Minutes', 'pct_exceeding': (sub['minutes_used'] > inc_min).mean() * 100})
    overage_records.append({'plan': plan, 'metric': 'Messages', 'pct_exceeding': (sub['message_count'] > inc_msg).mean() * 100})
    overage_records.append({'plan': plan, 'metric': 'Data (GB)', 'pct_exceeding': (sub['gb_used'] > inc_gb).mean() * 100})

overage_df = pd.DataFrame(overage_records)

overage_fig = px.bar(
    overage_df, x='metric', y='pct_exceeding', color='plan', barmode='group',
    title='Share of User-Months Exceeding the Plan Limit \u2014 the real revenue driver',
    labels={'pct_exceeding': '% of User-Months Exceeding Limit', 'metric': 'Usage Type', 'plan': 'Plan'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER},
    text=overage_df['pct_exceeding'].round(1).astype(str) + '%',
)
overage_fig.update_traces(textposition='outside')
overage_fig.update_layout(height=420, yaxis_ticksuffix='%')
overage_fig

### E. Monthly Trend (use the dropdown to switch metric)

In [139]:
trend_metrics = {
    'minutes_used': 'Average Minutes',
    'message_count': 'Average Messages',
    'gb_used': 'Average Data (GB)',
    'revenue': 'Average Revenue (USD)',
}
metric_keys = list(trend_metrics)

monthly_avg = user_activity.groupby(['plan', 'month'])[metric_keys].mean().reset_index()

trend_fig = go.Figure()
for m_idx, metric in enumerate(metric_keys):
    for plan in PLAN_ORDER:
        sub = monthly_avg[monthly_avg['plan'] == plan].sort_values('month')
        trend_fig.add_trace(go.Scatter(
            x=sub['month'], y=sub[metric], mode='lines+markers', name=plan.capitalize(),
            line=dict(color=PLAN_COLORS[plan]), visible=(m_idx == 0), legendgroup=plan,
        ))

n_plans = len(PLAN_ORDER)
buttons = []
for m_idx, metric in enumerate(metric_keys):
    visibility = [False] * (len(metric_keys) * n_plans)
    for p_idx in range(n_plans):
        visibility[m_idx * n_plans + p_idx] = True
    buttons.append(dict(label=trend_metrics[metric], method='update',
                         args=[{'visible': visibility}, {'yaxis': {'title': trend_metrics[metric]}}]))

trend_fig.update_layout(
    updatemenus=[dict(active=0, buttons=buttons, x=1.0, xanchor='right', y=1.18, yanchor='top')],
    title='Monthly Trend by Plan', xaxis_title='Month', yaxis_title=trend_metrics[metric_keys[0]],
    xaxis=dict(dtick=1), height=450,
)
trend_fig

### F. Revenue Distribution Snapshot

In [140]:
dist_fig = px.box(
    user_activity, x='plan', y='revenue', color='plan',
    title='Ultimate is stable, Surf is variable',
    labels={'plan': 'Plan', 'revenue': 'Revenue (USD)'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER},
)
dist_fig.update_layout(showlegend=False, height=420)
dist_fig

### G. Statistical Test Results

In [141]:
test_summary = pd.DataFrame({
    'Test': ['Surf vs. Ultimate Revenue', 'NY-NJ vs. Other Regions Revenue'],
    'H0': ['Mean revenue is equal', 'Mean revenue is equal'],
    'alpha': [0.05, 0.05],
    'p-value': ['3.17e-15', '0.0335'],
    'Decision': ['Reject H0', 'Reject H0'],
    'Business meaning': [
        'Plan choice has a statistically significant effect on revenue.',
        'Region has a statistically significant, though borderline, effect on revenue.',
    ],
})

stats_fig = go.Figure(data=[go.Table(
    header=dict(values=list(test_summary.columns), fill_color=PLAN_COLORS['ultimate'],
                font=dict(color='white', size=12), align='left'),
    cells=dict(values=[test_summary[c] for c in test_summary.columns],
               fill_color='white', align='left', height=28),
)])
stats_fig.update_layout(title="Statistical Test Summary (Welch's t-test, \u03b1 = 0.05)",
                         height=220, margin=dict(t=60, b=10))
stats_fig

## 8. User Behavior Analysis

### Calls

In [142]:
avg_minutes = user_activity.groupby(['plan', 'month'])['minutes_used'].mean().reset_index().round(2)
avg_minutes

,plan,month,minutes_used
0,surf,1,203.00
1,surf,2,297.00
2,surf,3,330.00
3,surf,4,351.54
4,surf,5,399.58
5,surf,6,431.30
6,surf,7,449.98
7,surf,8,410.11
8,surf,9,414.23
9,surf,10,429.73


In [143]:
fig = px.bar(
    avg_minutes, x='month', y='minutes_used', color='plan', barmode='group',
    title='Average Monthly Call Minutes by Plan',
    labels={'month': 'Month', 'minutes_used': 'Average Minutes', 'plan': 'Plan'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER})
fig.update_xaxes(type='category', categoryorder='array', categoryarray=[str(i) for i in range(1,13)])
fig

In [144]:
fig = px.histogram(
    user_activity, x='minutes_used', color='plan', barmode='overlay', nbins=30, opacity=0.7,
    title='Distribution of Monthly Call Minutes by Plan',
    labels={'minutes_used': 'Minutes per Month'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER},
)
fig.update_yaxes(title='Number of User-Months')
fig

In [145]:
# NOTE: aggregating 'minutes_used' (not 'calls_count') because this cell reports
# the mean/variance of monthly CALL DURATION, per the markdown instruction above.
call_duration_stats = user_activity.groupby('plan')['minutes_used'].agg(
    ['mean', 'median', 'std', 'var']
).round(2)
display(call_duration_stats)

,mean,median,std,var
plan,,,,
surf,428.75,425.0,234.45,54968.28
ultimate,430.45,424.0,240.51,57844.46


In [146]:
fig = px.box(user_activity, x='plan', y='minutes_used', color='plan',
    title='Boxplot of Monthly Call Minutes by Plan',
    labels={'plan': 'Plan', 'minutes_used': 'Minutes per Month'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER},
)
fig.update_layout(showlegend=False)
fig

[Formulate conclusions on how the users behave in terms of calling. Is their behaviour different between the plans?]

### Insight

**What we see:** Average monthly call usage is nearly identical between plans — Surf 428.75 minutes vs. Ultimate 430.45 minutes (~0.4% difference). The distribution is right-skewed with high variability (std ≈ 235–240 minutes) on both plans.

**What it means:** Users don't call more just because Ultimate offers a bigger allowance — actual talk-time habits are similar across the customer base regardless of plan.

**Business implication:** Despite similar usage, **36% of Surf users exceed the 500-minute limit** and pay overage charges, while **no Ultimate users exceed** the 3,000-minute allowance. The revenue gap between plans is therefore driven by *plan design*, not by fundamentally different calling behavior.

In [147]:
fig = px.histogram(
    user_activity, x='minutes_used', facet_col='plan', nbins=30, color='plan',
    title='Distribution of Monthly Call Minutes by Plan (mean vs median shows right-skew)',
    labels={'minutes_used': 'Minutes per Month'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER}
)
fig.update_yaxes(title='Number of User-Months', matches=None)
fig.update_xaxes(matches=None)
fig.update_layout(showlegend=False)
for i, plan in enumerate(PLAN_ORDER):
    col = i + 1
    data = user_activity.loc[user_activity['plan'] == plan, 'minutes_used']
    mean_val, median_val = data.mean(), data.median()
    skew_val = stats.skew(data.to_numpy())

    fig.add_vline(x=mean_val, line_dash='dash', line_color='red', col=col, row=1,
                   annotation_text=f'mean={mean_val:.0f}', annotation_position='top')
    fig.add_vline(x=median_val, line_dash='dot', line_color='black', col=col, row=1,
                   annotation_text=f'median={median_val:.0f}', annotation_position='bottom')
    fig.layout.annotations[i].text = f"{fig.layout.annotations[i].text} (skew={skew_val:.2f})"

fig

### Messages

In [148]:
user_activity.head()

,user_id,month,calls_count,minutes_used,message_count,used_internet,gb_used,plan,city,reg_date,churn_date,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,gb_per_month_included,revenue
0,1000,12,16.0,124.0,11.0,1901.47,2.0,ultimate,"Atlanta-Sandy Springs-Roswell, GA MSA",2018-12-24,NaT,1000,30720,3000,70,7,0.01,0.01,30.0,70.00
1,1001,8,27.0,182.0,30.0,6919.15,7.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0,20.00
2,1001,9,49.0,315.0,44.0,13314.82,14.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0,20.00
3,1001,10,65.0,393.0,53.0,22330.49,22.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0,90.09
4,1001,11,64.0,426.0,36.0,18504.30,19.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,NaT,50,15360,500,20,10,0.03,0.03,15.0,60.00


In [149]:
avg_messages = user_activity.groupby(['plan', 'month'])['message_count'].mean().reset_index()

In [150]:
fig = px.bar(
    avg_messages, x='month', y='message_count', color='plan', barmode='group',
    title='Average Monthly Messages by Plan',
    labels={'month': 'Month', 'message_count': 'Average Messages', 'plan': 'Plan'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER}
)
fig.update_xaxes(type='category', categoryorder='array', categoryarray=[str(i) for i in range(1,13)])
fig

In [151]:
fig = px.histogram(
    user_activity, x='gb_used', color='plan', barmode='overlay',
    nbins=30, opacity=0.7,
    title='Monthly Internet Usage by Plan',
    labels={'gb_used': 'GB per Month'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER}
)
fig.update_yaxes(title='Number of User-Months')
fig

In [152]:
fig = px.histogram(
    user_activity, x='message_count', color='plan', barmode='overlay', nbins=30, opacity=0.7,
    title='Distribution of Monthly Messages by Plan',
    labels={'message_count': 'Messages per Month'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER}
)
fig.update_yaxes(title='Number of User-Months')
fig

In [153]:
messages_stats = user_activity.groupby('plan')['message_count'].agg(['mean', 'var', 'std']).round(2)
print(messages_stats)


           mean      var    std
plan                           
surf      31.16  1126.72  33.57
ultimate  37.55  1208.76  34.77


In [154]:
fig = px.box(
    user_activity, x='plan', y='message_count', color='plan',
    title='Boxplot of Monthly Messages by Plan',
    labels={'plan': 'Plan', 'message_count': 'Messages per Month'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER},
)
fig.update_layout(showlegend=False)
fig

### Insight

**What we see:** Ultimate users send more messages on average — 37.55 vs. 31.16 per month (~20% higher). [FACT]

**What it means:** [INTERPRETATION] This gap is larger than the ~0.4% seen for calls, but still modest in absolute terms; it may reflect demographic differences between the two customer bases rather than the plan itself. [ASSUMPTION] Correlation with plan choice does not imply the plan *causes* more texting.

**Business implication:** **21.6% of Surf users exceed** the 50-message limit and pay extra, compared with **0% of Ultimate users** hitting the 1,000-message ceiling. As with calls, overage — not baseline usage — is what separates the two plans commercially.

### Internet

In [155]:
avg_gb = user_activity.groupby(['plan', 'month'])['gb_used'].mean().reset_index()

fig = px.bar(
    avg_gb, x='month', y='gb_used', color='plan', barmode='group',
    title='Average Monthly Internet Usage by Plan',
    labels={'month': 'Month', 'gb_used': 'Average GB Used', 'plan': 'Plan'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER},
)
fig.update_xaxes(type='category', categoryorder='array', categoryarray=[str(i) for i in range(1,13)])
fig

In [156]:
fig = px.histogram(
    user_activity, x='gb_used', color='plan', barmode='overlay', nbins=30, opacity=0.7,
    title='Distribution of Monthly Internet Usage by Plan',
    labels={'gb_used': 'GB per Month'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER},
)
fig.update_yaxes(title='Number of User-Months')
fig

In [157]:
gb_stats = user_activity.groupby('plan')['gb_used'].agg(['mean', 'var', 'std']).round(2)
print(gb_stats)


           mean    var   std
plan                        
surf      16.67  61.58  7.85
ultimate  17.31  58.83  7.67


In [158]:
fig = px.box(
    user_activity, x='plan', y='gb_used', color='plan',
    title='Boxplot of Monthly Internet Usage by Plan',
    labels={'plan': 'Plan', 'gb_used': 'GB per Month'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER},
)
fig.update_layout(showlegend=False)
fig

### Insight

**What we see:** Average monthly data usage is close between plans — Surf 16.67 GB vs. Ultimate 17.31 GB (~4% difference). [FACT]

**What it means:** [INTERPRETATION] Data is the metric where usage is most comparable across plans, yet it sits closest to Surf's included allowance, making it the most exposed limit.

**Business implication:** **57.9% of Surf users exceed** the 15 GB limit, versus only **5.7% of Ultimate users** exceeding 30 GB. Data overage is the single largest source of extra revenue among the three services, and the main reason Surf's revenue variance is so much higher than Ultimate's.

## 9. Revenue Analysis

In [159]:
revenue_stats = user_activity.groupby('plan')['revenue'].agg(['mean', 'var', 'std', 'sum', 'count']).round(2)
revenue_stats

,mean,var,std,sum,count
plan,,,,,
surf,60.71,3067.84,55.39,95491.18,1573
ultimate,72.31,129.85,11.40,52066.00,720


In [160]:
fig = px.histogram(
    user_activity, x='revenue', color='plan', barmode='overlay', nbins=30, opacity=0.7,
    title='Distribution of Monthly Revenue by Plan',
    labels={'revenue': 'Revenue (USD)'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER}
)
fig.update_yaxes(title='Number of User-Months')
fig

In [161]:
fig = px.box(
    user_activity, x='plan', y='revenue', color='plan',
    title='Boxplot of Monthly Revenue by Plan',
    labels={'plan': 'Plan', 'revenue': 'Revenue (USD)'},
    color_discrete_map=PLAN_COLORS, category_orders={'plan': PLAN_ORDER}
)
fig.update_layout(showlegend=False)
fig

### Insight

**What we see:** Average monthly revenue is higher for Ultimate ($72.31) than Surf ($60.71). Ultimate's revenue is tightly clustered around its $70 fixed fee (low variance); Surf's revenue is far more spread out due to overage charges (high variance).

**What it means:**  Ultimate is a more predictable revenue source per customer, while Surf's revenue depends heavily on how much a given user overshoots their limits.

**Business implication:** Surf still generates more **total** revenue ($95,491 vs. $52,066) because it has more user-months (1,573 vs. 720) — a larger, more price-sensitive customer base. Ultimate wins on revenue *per user*; Surf wins on revenue *at scale*. This distinction is the crux of the advertising-budget decision covered in [Section 13](#13-business-recommendation).

## 10. Statistical Hypothesis Testing

**H0:** mean revenue(Surf) = mean revenue(Ultimate) &nbsp;&nbsp;|&nbsp;&nbsp; **H1:** mean revenue(Surf) ≠ mean revenue(Ultimate)

In [162]:
# H0: mean revenue(Surf) == mean revenue(Ultimate)
# H1: mean revenue(Surf) != mean revenue(Ultimate)
# Two-sided Welch's t-test (equal_var=False) because the two plans show very different
# revenue variances (Surf's revenue is far more spread out than Ultimate's fixed-fee-dominated revenue).

alpha = 0.05

surf_revenue = user_activity.loc[user_activity['plan'] == 'surf', 'revenue']
ultimate_revenue = user_activity.loc[user_activity['plan'] == 'ultimate', 'revenue']

t_stat, p_value = stats.ttest_ind(surf_revenue, ultimate_revenue, equal_var=False)

print(f"Surf:     mean = {surf_revenue.mean():.2f} USD, var = {surf_revenue.var():.2f}, n = {len(surf_revenue)}")
print(f"Ultimate: mean = {ultimate_revenue.mean():.2f} USD, var = {ultimate_revenue.var():.2f}, n = {len(ultimate_revenue)}")
print(f"t-statistic = {t_stat:.4f}")
print(f"p-value     = {p_value:.2e}")

if p_value < alpha:
    print("Result: reject H0 -> average revenue differs significantly between Surf and Ultimate.")
else:
    print("Result: fail to reject H0 -> no significant difference detected.")

Surf:     mean = 60.71 USD, var = 3067.84, n = 1573
Ultimate: mean = 72.31 USD, var = 129.85, n = 720
t-statistic = -7.9521
p-value     = 3.17e-15
Result: reject H0 -> average revenue differs significantly between Surf and Ultimate.


**H0:** mean revenue(NY-NJ) = mean revenue(other regions) &nbsp;&nbsp;|&nbsp;&nbsp; **H1:** mean revenue(NY-NJ) ≠ mean revenue(other regions)

In [163]:
# H0: mean revenue(NY-NJ) == mean revenue(other regions)
# H1: mean revenue(NY-NJ) != mean revenue(other regions)
# Two-sided Welch's t-test (equal_var=False) since group sizes and variances are unequal.

alpha = 0.05

is_ny_nj = user_activity['city'].str.contains('NY-NJ', na=False)

ny_nj_revenue = user_activity.loc[is_ny_nj, 'revenue']
other_revenue = user_activity.loc[~is_ny_nj, 'revenue']

t_stat2, p_value2 = stats.ttest_ind(ny_nj_revenue, other_revenue, equal_var=False)

print(f"NY-NJ:  mean = {ny_nj_revenue.mean():.2f} USD, var = {ny_nj_revenue.var():.2f}, n = {len(ny_nj_revenue)}")
print(f"Other:  mean = {other_revenue.mean():.2f} USD, var = {other_revenue.var():.2f}, n = {len(other_revenue)}")
print(f"t-statistic = {t_stat2:.4f}")
print(f"p-value     = {p_value2:.4f}")

if p_value2 < alpha:
    print("Result: reject H0 -> average revenue in NY-NJ differs significantly from other regions.")
else:
    print("Result: fail to reject H0 -> no significant difference detected.")


NY-NJ:  mean = 59.92 USD, var = 1895.55, n = 377
Other:  mean = 65.22 USD, var = 2225.05, n = 1916
t-statistic = -2.1309
p-value     = 0.0335
Result: reject H0 -> average revenue in NY-NJ differs significantly from other regions.


## 11. General Conclusion

Overall Conclusion

**1. Data Preparation**: Date columns in all five tables were converted from string to datetime. No duplicates were found. About `19.5%` of calls and `13.1%` of internet sessions had zero usage and were retained. The `466` missing churn_date values indicate `active users`, not errors.

**2. User Behavior**: `Surf` and `Ultimate` users show similar usage patterns for `calls`, `messages`, and `data`. The differences are generally small, suggesting usage is driven more by actual communication needs than by plan limits.

**3. Plan Limits**: Despite similar usage, `36%` of Surf users exceed the call limit, `21.6%` exceed the message limit, and `57.9%` exceed the data limit, while almost no Ultimate users exceed their limits. This makes overage charges an important revenue source for Surf.

**4. Revenue**: Ultimate generates higher and more stable average revenue per user `($72.31 vs. $60.71)`. However, `Surf` generates higher total revenue `($95,491 vs. $52,066)` due to its larger `customer` base.

**5. Hypothesis 1** — `Surf vs. Ultimate Revenue`: Welch’s t-test produced `p = 3.17 × 10⁻¹⁵` `(< α = 0.05)`. H₀ is rejected, confirming a statistically significant difference in average revenue between the two plans.

**6. Hypothesis 2** — `NY-NJ` vs. Other Regions: Welch’s t-test produced p = 0.0335 (< α = 0.05). H₀ is rejected, indicating a statistically significant difference in average revenue between `NY-NJ ($59.92)` and other regions` ($65.22)`. However, the result is relatively close to the significance threshold, so more regional data would help confirm its stability.

> **7. Recommendation**: For higher revenue per user, prioritize `Ultimate`. For higher total revenue and market share,` Surf` remains attractive due to its larger user base and strong overage revenue potential.

## 12. Business Recommendation

| Objective | Recommended plan | Why |
|---|---|---|
| **Maximize revenue per user** | **Ultimate** | $72.31 avg. monthly revenue vs. $60.71 for Surf, with much lower variance — a more predictable per-customer return. |
| **Maximize total / market-scale revenue** | **Surf** | $95,491 vs. $52,066 total revenue, driven by a larger customer base (339 vs. 161 users) and strong overage income, especially from data. |
| **Statistical confidence** | Both differences are significant at α = 0.05 (Surf vs. Ultimate: p ≈ 3.17 × 10⁻¹⁵; NY-NJ vs. other regions: p = 0.0335) | The revenue gap is not due to random noise in this sample. |

**Practical takeaway for the advertising budget:** if the goal is efficient spend per acquired customer, prioritize **Ultimate**. If the goal is growing overall revenue and market share — leaning on Surf's larger addressable base and its overage-driven upside, particularly on data — **Surf** remains the stronger lever.

**Limitations to keep in mind:** this analysis is based on a single year (2018) and a relatively small, unevenly-sized sample (500 users; 80 in the NY-NJ region vs. 420 elsewhere). Statistical significance here does not imply the effect will hold at a different scale, in a different year, or that plan choice *causes* the usage patterns observed — only that the two are associated in this dataset. A follow-up analysis on more recent, larger, and better-balanced data is recommended before committing budget at scale.